In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma  # Changed 'chroma' to 'Chroma'

In [1]:
%pip install -qU langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [50]:
pip install google.generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 414.8 kB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 350.4 kB/s  0:00:45m0:00:0100:02
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [google.generativeai]ogle-ai-generativelanguage]
Note: you may need to restart the kernel to use updated packages.


In [45]:
pip install -U langchain-google-genai google-genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [27]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
import pandas as pd

In [12]:
books = pd.read_csv("books_cleaned.csv")

In [8]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_discription,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,0,Spider's Web : A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,0,I Am that : Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [13]:
books['tagged_description']

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [11]:
# building the vector search

books['tagged_description'].to_csv('tagged_description.txt', sep='\n',index=False,header=False)

In [5]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 2010, which is longer than the specified 2000
Created a chunk of size 2012, which is longer than the specified 2000
Created a chunk of size 2834, which is longer than the specified 2000
Created a chunk of size 2510, which is longer than the specified 2000
Created a chunk of size 2008, which is longer than the specified 2000
Created a chunk of size 2285, which is longer than the specified 2000
Created a chunk of size 2616, which is longer than the specified 2000
Created a chunk of size 2030, which is longer than the specified 2000
Created a chunk of size 2762, which is longer than the specified 2000
Created a chunk of size 2005, which is longer than the specified 2000
Created a chunk of size 2141, which is longer than the specified 2000
Created a chunk of size 2209, which is longer than the specified 2000
Created a chunk of size 2877, which is longer than the specified 2000
Created a chunk of size 2398, which is longer than the specified 2000
Created a chunk of s

In [22]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [56]:
import shutil
import os
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

In [59]:
pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.


In [61]:
%pip install sentence-transformers

  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 967.2 kB/s  0:00:00m-:--:--
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 553.6 kB/s  0:00:14 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━

In [7]:
import shutil
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Clear out the directory one more time
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

# 2. Re-init the local model
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Use a FRESH collection name to avoid any remaining locks
db_books = Chroma.from_documents(
    documents=documents, 
    embedding=embedding,
    persist_directory="./chroma_db",
    collection_name="new_book_index"  # <--- Changed this
)

print(f"✅ Success! {len(documents)} books embedded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Success! 1587 books embedded.


In [6]:
import shutil
import os
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Folder successfully deleted.")

In [18]:
query = "A Book to teach children about nature"

# k=5 means give me the top 5 most similar books
docs = db_books.similarity_search(query, k=5)

# Print the results
for i, doc in enumerate(docs):
    print(f"Recommendation {i+1}:")
    print(doc.page_content[:300] + "...") # Show the first 300 characters
    print("-" * 30)

Recommendation 1:
"9780618471164 The Earth and Its Peoples Brief Edition is a compact presentation of world history with a comparative approach and a global, balanced perspective. The themes of ""Environment and Technology"" and ""Diversity and Dominance"" unite the regions of the world. The Earth and Its Peoples Bri...
------------------------------
Recommendation 2:
"9780064404419 ""Culled from 69 stories collected in a [1930s] WPA project, [these 20] tales are organized into sections with themes like 'Tricksters' or 'Virtues and Vices,' each with a thoughtful introduction placing the individual stories in the context of feelings and background of the original ...
------------------------------
Recommendation 3:
9780762427321 Hawking's revolutionary look at the scientific discoveries that changed people's perceptions of the world now comes complete with color photographs and illustrations depicting theoretical models of the planets and their orbits.
9780763619312 When a third grade c

In [17]:
# .strip('"') removes leading/trailing double quotes specifically
isbn_str = docs[0].page_content.split()[0].strip().strip('"')
books[books['isbn13'] == int(isbn_str)]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_discription,title_and_subtitle,tagged_description
2922,9780618471164,0618471162,"Since 1500, Chapters 15-30",Richard W. Bulliet,Science,http://books.google.com/books/content?id=DS0C7...,The Earth and Its Peoples Brief Edition is a c...,2005.0,4.0,34.0,8.0,0,"Since 1500, Chapters 15-30 : The Earth and Its...",9780618471164 The Earth and Its Peoples Brief ...


In [24]:
def retrieve_semantic_recommendations(
    query:str,
    top_k:int = 10,
) -> pd.DataFrame:
     recs = db_books.similarity_search(query,k = 50)

     books_list = []

     for i in range(0,len(recs)):
         books_list += [int(recs[i].page_content.strip(' " ').split()[0])]

     return books[books['isbn13'].isin(books_list)].drop_duplicates(subset='isbn13').head(top_k)

In [23]:
retrieve_semantic_recommendations("A Book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,missing_discription,title_and_subtitle,tagged_description
59,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,0,The Family Way,9780007151240 It should be the most natural th...
104,9780060245603,0060245603,Today I Feel Silly & Other Moods That Make My Day,Jamie Lee Curtis,Juvenile Fiction,http://books.google.com/books/content?id=s_2GI...,Today I feel silly. Mom says it's the heat. I ...,1998.0,4.15,40.0,4030.0,0,Today I Feel Silly & Other Moods That Make My Day,9780060245603 Today I feel silly. Mom says it'...
270,9780060885373,0060885378,Little House in the Big Woods,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=7JctN...,A year in the life of two young girls growing ...,2007.0,4.18,198.0,193862.0,0,Little House in the Big Woods,9780060885373 A year in the life of two young ...
328,9780060974992,0060974990,Savage Inequalities,Jonathan Kozol,Education,http://books.google.com/books/content?id=UEJ3Q...,National Book Award-winning author Jonathan Ko...,1992.0,4.24,272.0,15021.0,0,Savage Inequalities : Children in America's Sc...,9780060974992 National Book Award-winning auth...
391,9780061205699,0061205699,To Kill a Mockingbird (slipcased edition),Harper Lee,Fiction,http://books.google.com/books/content?id=M9lKH...,"At the age of eight, Scout Finch is an entrenc...",2006.0,4.27,323.0,250.0,0,To Kill a Mockingbird (slipcased edition),"9780061205699 At the age of eight, Scout Finch..."
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
407,9780064404419,0064404412,The Rainbow People,Laurence Yep,Juvenile Fiction,http://books.google.com/books/content?id=5AHwq...,"""Culled from 69 stories collected in a [1930s]...",1992.0,3.75,208.0,202.0,0,The Rainbow People,"9780064404419 ""Culled from 69 stories collecte..."
411,9780064405478,0064405478,Betsy Was a Junior,Maud Hart Lovelace,Juvenile Fiction,http://books.google.com/books/content?id=TNcQQ...,A small town shortly after the turn of the cen...,1995.0,4.17,320.0,4744.0,0,Betsy Was a Junior,9780064405478 A small town shortly after the t...
415,9780064406307,006440630X,The Midwife's Apprentice (rpkg),Karen Cushman,Juvenile Fiction,http://books.google.com/books/content?id=Bhm76...,"'Like Cushman's 1995 Newbery Honor Book, Cathe...",1996.0,3.72,128.0,35319.0,0,The Midwife's Apprentice (rpkg),9780064406307 'Like Cushman's 1995 Newbery Hon...
428,9780064434942,006443494X,A Little House Birthday,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=fyfDR...,Join the Ingalls family as they celebrate litt...,1998.0,4.13,40.0,444.0,0,A Little House Birthday,9780064434942 Join the Ingalls family as they ...


In [25]:
# Testing the semantic "vibe" of the system
challenge_results = retrieve_semantic_recommendations("A story about overcoming fear and finding courage")
challenge_results[['title', 'authors', 'categories']]

,title,authors,categories
101,Identity,Milan Kundera,Fiction
132,Warrior of the Light,Paulo Coelho,Fiction
195,Touching the Void,Joe Simpson,Sports & Recreation
214,Charms for the Easy Life,Kaye Gibbons,Fiction
224,Brave New World and Brave New World Revisited,Aldous Huxley,Fiction
274,The Alchemist - Gift Edition,Paulo Coelho,Fiction
301,Identity,Milan Kundera,Fiction
305,The Map That Changed the World,Simon Winchester;Soun Vannithone,Biography & Autobiography
336,Prisoner's Dilemma,Richard Powers,Fiction
366,The Alchemist,Paulo Coelho,Fiction


In [26]:
from IPython.display import Image, display, HTML

def display_recommendations_with_images(query, top_k=5):
    df = retrieve_semantic_recommendations(query, top_k=top_k)
    
    for _, row in df.iterrows():
        display(HTML(f"<h3>{row['title']}</h3>"))
        display(HTML(f"<b>Author:</b> {row['authors']} | <b>Rating:</b> {row['average_rating']}"))
        display(Image(url=row['thumbnail'], width=100))
        print("-" * 50)

# Try it out!
display_recommendations_with_images("A Book to teach children about nature")

--------------------------------------------------


--------------------------------------------------


--------------------------------------------------


--------------------------------------------------


--------------------------------------------------
